In [1]:
from pathlib import Path
import pandas as pd

In [2]:
download_data_dir = Path("downloaded_data/")
raw_data_dir = Path("raw_data/")
dataset_name = "wmt24_chat_dev"

# Preprocess data

## Create raw data per segment

In [8]:
for lp in ["en-de", "en-fr", "en-pt-br", "en-ko", "en-nl"]:
    print(f"{lp}")
    if lp != "en-pt-br":
        eng, non_eng = lp.split("-")
    else:
        eng, non_eng = "en", "pt-br"
    df = pd.read_csv(
        download_data_dir / f"{dataset_name}.{lp}.csv",
        keep_default_na=False,
        na_values=["NaN"],
    )
    # rename for tower_eval purposes
    df.rename(
        columns={
            "source": "src",
            "reference": "ref",
        },
        inplace=True,
    )
    # full
    save_dir = Path(f"{raw_data_dir}/mt/{dataset_name}.{non_eng}")
    save_dir.mkdir(parents=True, exist_ok=True)
    df.to_json(
        save_dir / f"test.jsonl", orient="records", lines=True, force_ascii=False
    )
    # en-xx direction
    en_xx_df = df[df["source_language"] == "en"].reset_index(drop=True)
    save_dir = Path(f"{raw_data_dir}/mt/{dataset_name}.{eng}-{non_eng}")
    save_dir.mkdir(parents=True, exist_ok=True)
    en_xx_df.to_json(
        save_dir / f"test.jsonl", orient="records", lines=True, force_ascii=False
    )
    # xx-en direction
    xx_en_df = df[df["target_language"] == "en"].reset_index(drop=True)
    save_dir = Path(f"{raw_data_dir}/mt/{dataset_name}.{non_eng}-{eng}")
    save_dir.mkdir(parents=True, exist_ok=True)
    xx_en_df.to_json(
        save_dir / f"test.jsonl", orient="records", lines=True, force_ascii=False
    )

en-de
en-fr
en-pt-br
en-ko
en-nl


## Create tower instructions

In [4]:
from transformers import AutoTokenizer
from tower_eval.utils import write_lines

/mnt/data/jpombal/tower-eval/tower-eval-env/lib/python3.10/site-packages/transformers/utils/hub.py:124: FutureWarning: Using `TRANSFORMERS_CACHE` is deprecated and will be removed in v5 of Transformers. Use `HF_HOME` instead.
  warnings.warn(


In [10]:
t = AutoTokenizer.from_pretrained("Unbabel/TowerInstruct-13B-v0.2")

Special tokens have been added in the vocabulary, make sure the associated word embeddings are fine-tuned or trained.


## No context

In [11]:
def apply_instruction(src, src_lang, tgt_lang, tokenizer):
    messages = [
        {
            "role": "user",
            "content": f"Translate the following {src_lang} source text to {tgt_lang}:\n{src_lang}: {src}\n{tgt_lang}: ",
        }
    ]
    return tokenizer.apply_chat_template(
        messages, tokenize=False, add_generation_prompt=True
    )

In [12]:
instructions_data_dir = Path("instructions/no_context")
code_lang_dict = {
    "en": "English",
    "de": "German",
    "fr": "French",
    "pt-br": "Brazilian Portuguese",
    "ko": "Korean",
    "nl": "Dutch",
}

lps = [f"en-{xx}" for xx in code_lang_dict.keys() if xx != "en"]

for lp in lps:
    print(lp)
    if lp == "en-pt-br":
        non_eng = "pt-br"
        eng = "en"
    else:
        eng, non_eng = lp.split("-")

    # en-xx direction
    save_dir = Path(f"{instructions_data_dir}/mt/{dataset_name}.{eng}-{non_eng}")
    df = pd.read_json(
        raw_data_dir / f"mt/{dataset_name}.{eng}-{non_eng}/test.jsonl", lines=True
    )
    instructions = (
        df["src"]
        .apply(
            lambda x: apply_instruction(
                x, code_lang_dict[eng], code_lang_dict[non_eng], t
            )
        )
        .tolist()
    )
    write_lines(save_dir / "instructions.txt", instructions, escape_newline=True)
    # xx-en direction
    save_dir = Path(f"{instructions_data_dir}/mt/{dataset_name}.{non_eng}-{eng}")
    df = pd.read_json(
        raw_data_dir / f"mt/{dataset_name}.{non_eng}-{eng}/test.jsonl", lines=True
    )
    instructions = (
        df["src"]
        .apply(
            lambda x: apply_instruction(
                x, code_lang_dict[non_eng], code_lang_dict[eng], t
            )
        )
        .tolist()
    )
    write_lines(save_dir / "instructions.txt", instructions, escape_newline=True)

en-de
en-fr
en-pt-br
en-ko
en-nl


## Full context

In [3]:
from pathlib import Path
import pandas as pd

from transformers import AutoTokenizer
from tower_eval.utils import write_lines

/mnt/data/jpombal/tower-eval/tower-eval-env/lib/python3.10/site-packages/transformers/utils/hub.py:124: FutureWarning: Using `TRANSFORMERS_CACHE` is deprecated and will be removed in v5 of Transformers. Use `HF_HOME` instead.
  warnings.warn(


In [4]:
def apply_instruction(src, src_lang, tgt_lang, tokenizer):
    messages = [
        {
            "role": "user",
            "content": f"Translate the following {src_lang} source text to {tgt_lang}:\n{src_lang}: {src}\n{tgt_lang}: ",
        }
    ]
    return tokenizer.apply_chat_template(
        messages, tokenize=False, add_generation_prompt=True
    )


def apply_series_of_instruction(
    src,
    src_lang,
    tgt_lang,
    tokenizer,
    previous_sources,
    previous_directions,
    current_direction,
):
    initial_string = f"Context: "
    for s, d in zip(previous_sources, previous_directions):
        if d == current_direction:
            l = src_lang
        else:
            l = tgt_lang
        initial_string += f"{s}\n"
    final_string = (
        initial_string
        + f"\nTranslate the {src_lang} source text to {tgt_lang}, given the context."
    )
    final_string = final_string + (f"\n{src_lang}: {src}\n{tgt_lang}: ")
    messages = [
        {
            "role": "user",
            "content": final_string,
        }
    ]
    return tokenizer.apply_chat_template(
        messages, tokenize=False, add_generation_prompt=True
    )

In [5]:
t = AutoTokenizer.from_pretrained("Unbabel/TowerInstruct-13B-v0.2")

Special tokens have been added in the vocabulary, make sure the associated word embeddings are fine-tuned or trained.


In [6]:
instructions_data_dir = Path("instructions/full_context")
code_lang_dict = {
    "en": "English",
    "de": "German",
    "fr": "French",
    "pt-br": "Brazilian Portuguese",
    "ko": "Korean",
    "nl": "Dutch",
}

lps = [f"en-{xx}" for xx in code_lang_dict.keys() if xx != "en"]

for lp in lps:
    print(lp)
    if lp == "en-pt-br":
        non_eng = "pt-br"
        eng = "en"
    else:
        eng, non_eng = lp.split("-")

    instructions = []
    df = pd.read_json(
        f"/mnt/data/jpombal/wmt24-chat-translation/raw_data/mt/{dataset_name}.{non_eng}/test.jsonl",
        lines=True,
    )
    register_convo = False
    last_i = 0
    for row_i, row in df.iterrows():
        convo_doc_id = row["doc_id"]
        if row_i != len(df) - 1:
            next_doc_id = df.loc[row_i + 1, "doc_id"]
            if convo_doc_id != next_doc_id:
                register_convo = True
                convo_df = df.loc[last_i:row_i].reset_index(drop=True)
                last_i = row_i + 1
        else:
            register_convo = True
            convo_df = df.loc[last_i:].reset_index(drop=True)
        if register_convo:
            for i, row in convo_df.iterrows():
                src_lang = code_lang_dict[row["source_language"]]
                tgt_lang = code_lang_dict[row["target_language"]]
                if (
                    row["doc_id"] == "646db6b7c49c6826f405703a"
                    and row["src"] == "Hello !"
                ):
                    a = 1
                if i == 0:
                    instructions.append(
                        apply_instruction(
                            row["src"],
                            src_lang,
                            tgt_lang,
                            t,
                        )
                    )
                else:
                    previous_sources = convo_df.loc[: i - 1, "src"].tolist()
                    previous_directions = convo_df.loc[: i - 1, "sender"].tolist()
                    instructions.append(
                        apply_series_of_instruction(
                            row["src"],
                            src_lang,
                            tgt_lang,
                            t,
                            previous_sources=previous_sources,
                            previous_directions=previous_directions,
                            current_direction=row["sender"],
                        )
                    )
        register_convo = False

    df["instructions"] = instructions
    # en-xx direction
    save_dir = Path(f"{instructions_data_dir}/mt/{dataset_name}.{eng}-{non_eng}")
    instructions = df[df["source_language"] == "en"]["instructions"].tolist()
    write_lines(save_dir / "instructions.txt", instructions, escape_newline=True)
    # xx-en direction
    save_dir = Path(f"{instructions_data_dir}/mt/{dataset_name}.{non_eng}-{eng}")
    instructions = df[df["target_language"] == "en"]["instructions"].tolist()
    write_lines(save_dir / "instructions.txt", instructions, escape_newline=True)

en-de
en-fr
en-pt-br
en-ko
en-nl


# Empty system prompt

In [4]:
t = AutoTokenizer.from_pretrained(
    "/mnt/data/jpombal/axolotl/TowerInstruct_7B_chat_ckpts_new/checkpoint-5624"
)

Special tokens have been added in the vocabulary, make sure the associated word embeddings are fine-tuned or trained.


In [5]:
def apply_instruction(src, src_lang, tgt_lang, tokenizer):
    messages = [
        {
            "role": "system",
            "content": "",
        },
        {
            "role": "user",
            "content": f"Translate the following {src_lang} source text to {tgt_lang}:\n{src_lang}: {src}\n{tgt_lang}: ",
        },
    ]
    return tokenizer.apply_chat_template(
        messages, tokenize=False, add_generation_prompt=True
    )


def apply_series_of_instruction(
    src,
    src_lang,
    tgt_lang,
    tokenizer,
    previous_sources,
    previous_directions,
    current_direction,
):
    initial_string = f"Context: "
    for s, d in zip(previous_sources, previous_directions):
        if d == current_direction:
            l = src_lang
        else:
            l = tgt_lang
        initial_string += f"{s}\n"
    final_string = (
        initial_string
        + f"\nTranslate the {src_lang} source text to {tgt_lang}, given the context."
    )
    final_string = final_string + (f"\n{src_lang}: {src}\n{tgt_lang}: ")
    messages = [
        {
            "role": "system",
            "content": "",
        },
        {
            "role": "user",
            "content": final_string,
        },
    ]
    return tokenizer.apply_chat_template(
        messages, tokenize=False, add_generation_prompt=True
    )

In [6]:
instructions_data_dir = Path("instructions/no_context_empty_sys")
code_lang_dict = {
    "en": "English",
    "de": "German",
    "fr": "French",
    "pt-br": "Brazilian Portuguese",
    "ko": "Korean",
    "nl": "Dutch",
}

lps = [f"en-{xx}" for xx in code_lang_dict.keys() if xx != "en"]

for lp in lps:
    print(lp)
    if lp == "en-pt-br":
        non_eng = "pt-br"
        eng = "en"
    else:
        eng, non_eng = lp.split("-")

    # en-xx direction
    save_dir = Path(f"{instructions_data_dir}/mt/{dataset_name}.{eng}-{non_eng}")
    df = pd.read_json(
        raw_data_dir / f"mt/{dataset_name}.{eng}-{non_eng}/test.jsonl", lines=True
    )
    instructions = (
        df["src"]
        .apply(
            lambda x: apply_instruction(
                x, code_lang_dict[eng], code_lang_dict[non_eng], t
            )
        )
        .tolist()
    )
    write_lines(
        save_dir / "instructions.txt", instructions, escape_newline=True, verbose=False
    )
    # xx-en direction
    save_dir = Path(f"{instructions_data_dir}/mt/{dataset_name}.{non_eng}-{eng}")
    df = pd.read_json(
        raw_data_dir / f"mt/{dataset_name}.{non_eng}-{eng}/test.jsonl", lines=True
    )
    instructions = (
        df["src"]
        .apply(
            lambda x: apply_instruction(
                x, code_lang_dict[non_eng], code_lang_dict[eng], t
            )
        )
        .tolist()
    )
    write_lines(
        save_dir / "instructions.txt", instructions, escape_newline=True, verbose=False
    )

en-de
en-fr
en-pt-br
en-ko
en-nl


In [7]:
instructions_data_dir = Path("instructions/full_context_empty_sys")
code_lang_dict = {
    "en": "English",
    "de": "German",
    "fr": "French",
    "pt-br": "Brazilian Portuguese",
    "ko": "Korean",
    "nl": "Dutch",
}

lps = [f"en-{xx}" for xx in code_lang_dict.keys() if xx != "en"]

for lp in lps:
    print(lp)
    if lp == "en-pt-br":
        non_eng = "pt-br"
        eng = "en"
    else:
        eng, non_eng = lp.split("-")

    instructions = []
    df = pd.read_json(
        f"/mnt/data/jpombal/wmt24-chat-translation/raw_data/mt/{dataset_name}.{non_eng}/test.jsonl",
        lines=True,
    )
    register_convo = False
    last_i = 0
    for row_i, row in df.iterrows():
        convo_doc_id = row["doc_id"]
        if row_i != len(df) - 1:
            next_doc_id = df.loc[row_i + 1, "doc_id"]
            if convo_doc_id != next_doc_id:
                register_convo = True
                convo_df = df.loc[last_i:row_i].reset_index(drop=True)
                last_i = row_i + 1
        else:
            register_convo = True
            convo_df = df.loc[last_i:].reset_index(drop=True)
        if register_convo:
            for i, row in convo_df.iterrows():
                src_lang = code_lang_dict[row["source_language"]]
                tgt_lang = code_lang_dict[row["target_language"]]
                if (
                    row["doc_id"] == "646db6b7c49c6826f405703a"
                    and row["src"] == "Hello !"
                ):
                    a = 1
                if i == 0:
                    instructions.append(
                        apply_instruction(
                            row["src"],
                            src_lang,
                            tgt_lang,
                            t,
                        )
                    )
                else:
                    previous_sources = convo_df.loc[: i - 1, "src"].tolist()
                    previous_directions = convo_df.loc[: i - 1, "sender"].tolist()
                    instructions.append(
                        apply_series_of_instruction(
                            row["src"],
                            src_lang,
                            tgt_lang,
                            t,
                            previous_sources=previous_sources,
                            previous_directions=previous_directions,
                            current_direction=row["sender"],
                        )
                    )
        register_convo = False

    df["instructions"] = instructions
    # en-xx direction
    save_dir = Path(f"{instructions_data_dir}/mt/{dataset_name}.{eng}-{non_eng}")
    instructions = df[df["source_language"] == "en"]["instructions"].tolist()
    write_lines(
        save_dir / "instructions.txt", instructions, escape_newline=True, verbose=False
    )
    # xx-en direction
    save_dir = Path(f"{instructions_data_dir}/mt/{dataset_name}.{non_eng}-{eng}")
    instructions = df[df["target_language"] == "en"]["instructions"].tolist()
    write_lines(
        save_dir / "instructions.txt", instructions, escape_newline=True, verbose=False
    )

en-de
en-fr
en-pt-br
en-ko
en-nl


# Empty system llama3 prompt

In [5]:
t = AutoTokenizer.from_pretrained("Unbabel/Tower-Llama3-70B-WMT24-Merged-2106")

Special tokens have been added in the vocabulary, make sure the associated word embeddings are fine-tuned or trained.


In [8]:
def apply_instruction(src, src_lang, tgt_lang, tokenizer):
    return f"<|start_header_id|>system<|end_header_id|>\n\n<|eot_id|><|start_header_id|>user<|end_header_id|>\n\nTranslate the following {src_lang} source text to {tgt_lang}:\n{src_lang}: {src}\n{tgt_lang}: <|eot_id|><|start_header_id|>assistant<|end_header_id|>\n\n"


def apply_series_of_instruction(
    src,
    src_lang,
    tgt_lang,
    tokenizer,
    previous_sources,
    previous_directions,
    current_direction,
):
    initial_string = f"Context: "
    for s, d in zip(previous_sources, previous_directions):
        if d == current_direction:
            l = src_lang
        else:
            l = tgt_lang
        initial_string += f"{s}\n"
    final_string = (
        initial_string
        + f"\nTranslate the {src_lang} source text to {tgt_lang}, given the context."
    )
    final_string = final_string + (f"\n{src_lang}: {src}\n{tgt_lang}: ")

    return f"<|start_header_id|>system<|end_header_id|>\n\n<|eot_id|><|start_header_id|>user<|end_header_id|>\n\n{final_string}<|eot_id|><|start_header_id|>assistant<|end_header_id|>\n\n"

In [10]:
instructions_data_dir = Path("instructions/no_context_empty_sys_llama3")
code_lang_dict = {
    "en": "English",
    "de": "German",
    "fr": "French",
    "pt-br": "Brazilian Portuguese",
    "ko": "Korean",
    "nl": "Dutch",
}

lps = [f"en-{xx}" for xx in code_lang_dict.keys() if xx != "en"]

for lp in lps:
    print(lp)
    if lp == "en-pt-br":
        non_eng = "pt-br"
        eng = "en"
    else:
        eng, non_eng = lp.split("-")

    # en-xx direction
    save_dir = Path(f"{instructions_data_dir}/mt/{dataset_name}.{eng}-{non_eng}")
    df = pd.read_json(
        raw_data_dir / f"mt/{dataset_name}.{eng}-{non_eng}/test.jsonl", lines=True
    )
    instructions = (
        df["src"]
        .apply(
            lambda x: apply_instruction(
                x, code_lang_dict[eng], code_lang_dict[non_eng], t
            )
        )
        .tolist()
    )
    write_lines(
        save_dir / "instructions.txt", instructions, escape_newline=True, verbose=False
    )
    # xx-en direction
    save_dir = Path(f"{instructions_data_dir}/mt/{dataset_name}.{non_eng}-{eng}")
    df = pd.read_json(
        raw_data_dir / f"mt/{dataset_name}.{non_eng}-{eng}/test.jsonl", lines=True
    )
    instructions = (
        df["src"]
        .apply(
            lambda x: apply_instruction(
                x, code_lang_dict[non_eng], code_lang_dict[eng], t
            )
        )
        .tolist()
    )
    write_lines(
        save_dir / "instructions.txt", instructions, escape_newline=True, verbose=False
    )

en-de
en-fr
en-pt-br
en-ko
en-nl


In [12]:
instructions_data_dir = Path("instructions/full_context_empty_sys_llama3")
code_lang_dict = {
    "en": "English",
    "de": "German",
    "fr": "French",
    "pt-br": "Brazilian Portuguese",
    "ko": "Korean",
    "nl": "Dutch",
}

lps = [f"en-{xx}" for xx in code_lang_dict.keys() if xx != "en"]

for lp in lps:
    print(lp)
    if lp == "en-pt-br":
        non_eng = "pt-br"
        eng = "en"
    else:
        eng, non_eng = lp.split("-")

    instructions = []
    df = pd.read_json(
        f"/mnt/data/jpombal/wmt24-chat-translation/raw_data/mt/{dataset_name}.{non_eng}/test.jsonl",
        lines=True,
    )
    register_convo = False
    last_i = 0
    for row_i, row in df.iterrows():
        convo_doc_id = row["doc_id"]
        if row_i != len(df) - 1:
            next_doc_id = df.loc[row_i + 1, "doc_id"]
            if convo_doc_id != next_doc_id:
                register_convo = True
                convo_df = df.loc[last_i:row_i].reset_index(drop=True)
                last_i = row_i + 1
        else:
            register_convo = True
            convo_df = df.loc[last_i:].reset_index(drop=True)
        if register_convo:
            for i, row in convo_df.iterrows():
                src_lang = code_lang_dict[row["source_language"]]
                tgt_lang = code_lang_dict[row["target_language"]]
                if (
                    row["doc_id"] == "646db6b7c49c6826f405703a"
                    and row["src"] == "Hello !"
                ):
                    a = 1
                if i == 0:
                    instructions.append(
                        apply_instruction(
                            row["src"],
                            src_lang,
                            tgt_lang,
                            t,
                        )
                    )
                else:
                    previous_sources = convo_df.loc[: i - 1, "src"].tolist()
                    previous_directions = convo_df.loc[: i - 1, "sender"].tolist()
                    instructions.append(
                        apply_series_of_instruction(
                            row["src"],
                            src_lang,
                            tgt_lang,
                            t,
                            previous_sources=previous_sources,
                            previous_directions=previous_directions,
                            current_direction=row["sender"],
                        )
                    )
        register_convo = False

    df["instructions"] = instructions
    # en-xx direction
    save_dir = Path(f"{instructions_data_dir}/mt/{dataset_name}.{eng}-{non_eng}")
    instructions = df[df["source_language"] == "en"]["instructions"].tolist()
    write_lines(
        save_dir / "instructions.txt", instructions, escape_newline=True, verbose=False
    )
    # xx-en direction
    save_dir = Path(f"{instructions_data_dir}/mt/{dataset_name}.{non_eng}-{eng}")
    instructions = df[df["target_language"] == "en"]["instructions"].tolist()
    write_lines(
        save_dir / "instructions.txt", instructions, escape_newline=True, verbose=False
    )

en-de
en-fr
en-pt-br
en-ko
en-nl
